In [ ]:
import os
import time
from shutil import which

from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

NOMBRE_GRUPO = "energia-y-sustentabilidad-1"
MONGO_URI = "mongodb://database:27017/"

client = MongoClient(MONGO_URI)
db = client["TiendaBigData"]
coleccion = db["AmazonLaptops"]

print("Conexion exitosa a MongoDB")

In [ ]:
binary = which("brave-browser") or which("google-chrome") or "/usr/bin/brave-browser"

options = Options()
options.binary_location = binary
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1366,768")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=options)
productos = []

try:
    driver.get("https://www.amazon.es/s?k=laptops")
    time.sleep(5)

    if "captcha" in driver.page_source.lower() or "robot" in driver.page_source.lower():
        print("Si ves un captcha, resuelvelo en noVNC y luego vuelve a ejecutar esta celda.")

    WebDriverWait(driver, 25).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div[data-component-type='s-search-result']"))
    )

    bloques = driver.find_elements(By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
    fecha_captura = time.strftime("%Y-%m-%d %H:%M:%S")

    for bloque in bloques[:20]:
        try:
            identificador = bloque.find_element(By.CSS_SELECTOR, "h2 span").text.strip()
            precio_entero = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text
            precio_decimal = bloque.find_element(By.CSS_SELECTOR, ".a-price-fraction").text
            entero_limpio = precio_entero.replace(".", "").replace(",", "").strip()
            decimal_limpio = precio_decimal.replace(".", "").replace(",", "").strip()
            valor = float(f"{entero_limpio}.{decimal_limpio}")

            productos.append({
                "identificador": identificador,
                "valor": valor,
                "grupo": NOMBRE_GRUPO,
                "fecha_captura": fecha_captura
            })
        except Exception:
            continue
finally:
    driver.quit()

print(f"Productos capturados: {len(productos)}")

In [ ]:
coleccion.delete_many({"grupo": NOMBRE_GRUPO})

if productos:
    coleccion.insert_many(productos)
    print("Datos cargados correctamente en MongoDB")
    print(productos[0])
else:
    print("No se capturaron productos. Revisa la pagina o el captcha.")